In [8]:
import spacy

nlp = spacy.blank("en")
doc = nlp("Hello World!")

print(*(token.text for token in doc), sep="\n")

# for token in doc:
#     print(token.text)

Hello
World
!


In [10]:
type(doc)

spacy.tokens.doc.Doc

In [14]:
type(doc[1])

spacy.tokens.token.Token

In [17]:
type(doc[1].text)

str

In [ ]:
# A slice from the Doc is a Span object
span = doc[1:3]

# Get the span text via the .text attribute
print(span.text)

world!


i is the index of the token within the parent document.

text returns the token text.

is_alpha, is_punct and like_num return boolean values indicating whether the token consists of alphabetic characters, whether it's punctuation or whether it resembles a number. For example, a token "10" – one, zero – or the word "ten" – T, E, N.

These attributes are also called lexical attributes: they refer to the entry in the vocabulary and don't depend on the token's context.


In [26]:
doc = nlp("It costs $5 (five dollars).")
print("Index:   ", [token.i for token in doc])
print("Text:    ", [token.text for token in doc])

print("is_alpha:", [token.is_alpha for token in doc])
print("is_punct:", [token.is_punct for token in doc])
print("like_num:", [token.like_num for token in doc])

Index:    [0, 1, 2, 3, 4, 5, 6, 7, 8]
Text:     ['It', 'costs', '$', '5', '(', 'five', 'dollars', ')', '.']
is_alpha: [True, True, False, False, False, True, True, False, False]
is_punct: [False, False, False, False, True, False, False, True, True]
like_num: [False, False, False, True, False, True, False, False, False]


## POS Tags


In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp("She ate the pizza")

for token in doc:
    print(token.text, token.pos_)

She PRON
ate VERB
the DET
pizza NOUN


## Syntatic Dependencies

- .head = Who is my parent?
- .dep\_ = What is my relationship with my parent?


In [ ]:
for token in doc:
    print(token.text, token.pos_, token.dep_, token.head)

She PRON nsubj ate
ate VERB ROOT ate
the DET det pizza
pizza NOUN dobj ate


## Named

Named entities are "real world objects" that are assigned a name – for example, a person, an organization or a country.

The `doc.ents` property lets you access the named entities predicted by the named entity recognition model.

It returns an iterator of Span objects, so we can print the entity text and the entity label using the `.label_` attribute.


In [ ]:
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")

for ent in doc.ents:
    print(ent.text, ent.label_)

    # Get quick definitions of the most common tags and labels.
    print(spacy.explain(ent.label_))

Apple ORG
Companies, agencies, institutions, etc.
U.K. GPE
Countries, cities, states
$1 billion MONEY
Monetary values, including unit


## Pattern Matching

To use a pattern, we first import the matcher from `spacy.matcher`.

We also load a pipeline and create the `nlp` object.

The matcher is initialized with the shared vocabulary, `nlp.vocab`. You'll learn more about this later – for now, just remember to always pass it in.

The `matcher.add` method lets you add a pattern. The first argument is a unique ID to identify which pattern was matched. The second argument is a list of patterns.

To match the pattern on a text, we can call the matcher on any doc.

- *When you call the matcher on a doc, it returns a list of tuples.Each tuple consists of three values: the **match ID, the start index and the end index** of the matched span.*


In [37]:
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

pattern = [{"TEXT": "iPhone"}, {"TEXT": "X"}]
matcher.add("IPHONE_PATTERN", [pattern])

doc = nlp("Upcoming iPhone X release date leaked")
matches = matcher(doc)

for match_id, start, end in matches:
    matched_span = doc[start:end]
    print(matched_span.text)

iPhone X


In [39]:
pattern = [
    {"IS_DIGIT": True},
    {"LOWER": "fifa"},
    {"LOWER": "world"},
    {"LOWER": "cup"},
    {"IS_PUNCT": True},
]
matcher.add("FIFA PATTERN", [pattern])

doc = nlp("2018 FIFA World Cup: France won!")
matches = matcher(doc)

for match_id, start, end in matches:
    matched_span = doc[start:end]
    print(matched_span.text)

2018 FIFA World Cup:


In [40]:
pattern = [{"LEMMA": "love", "POS": "VERB"}, {"POS": "NOUN"}]
matcher.add("DOG&CAT PATTERN", [pattern])

doc = nlp("I loved dogs but now I love cats more.")
matches = matcher(doc)

for match_id, start, end in matches:
    matched_span = doc[start:end]
    print(matched_span.text)

loved dogs
love cats
